# 03 — Database Setup

Load the processed HDB resale transaction dataset into DuckDB for SQL-based business analysis.

In [1]:
from pathlib import Path

PROJECT_ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data").is_dir() and (path / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project directory.")

DATABASE_PATH = PROJECT_ROOT / "database" / "hdb.duckdb"
DATABASE_PATH.parent.mkdir(parents=True, exist_ok=True)

import duckdb
import pandas as pd

In [2]:
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "hdb_resale_clean.csv")

In [3]:
df.shape

(239330, 22)

In [4]:
con = duckdb.connect(str(DATABASE_PATH))

In [5]:
con.execute("""
    CREATE OR REPLACE TABLE resale_transactions AS
    SELECT * FROM df
""")

In [6]:
con.execute("SHOW TABLES").fetchdf()

,name
0,resale_transactions


In [7]:
con.execute("""
    SELECT COUNT(*) AS total_transactions
    FROM resale_transactions
""").fetchdf()

,total_transactions
0,239330


In [8]:
con.execute("""
    SELECT *
    FROM resale_transactions
    LIMIT 5
""").fetchdf()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,quarter,price_per_sqm,flat_age,storey_lower,storey_upper,storey_midpoint,remaining_lease_years,remaining_lease_extra_months,remaining_lease_months,remaining_lease_years_numeric
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,...,2017Q1,5272.73,38,10,12,11.0,61,4,736,61.33
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,...,2017Q1,3731.34,39,1,3,2.0,60,7,727,60.58
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,2017Q1,3910.45,37,1,3,2.0,62,5,749,62.42
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,...,2017Q1,3897.06,37,4,6,5.0,62,1,745,62.08
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,2017Q1,3955.22,37,1,3,2.0,62,5,749,62.42


In [9]:
con.execute("""
    DESCRIBE resale_transactions
""").fetchdf()

,column_name,column_type,null,key,default,extra
0,month,VARCHAR,YES,None,None,None
1,town,VARCHAR,YES,None,None,None
2,flat_type,VARCHAR,YES,None,None,None
3,block,VARCHAR,YES,None,None,None
4,street_name,VARCHAR,YES,None,None,None
5,storey_range,VARCHAR,YES,None,None,None
6,floor_area_sqm,DOUBLE,YES,None,None,None
7,flat_model,VARCHAR,YES,None,None,None
8,lease_commence_date,BIGINT,YES,None,None,None
9,remaining_lease,VARCHAR,YES,None,None,None


In [10]:
con.execute("""
    SELECT
        town,
        COUNT(*) AS transactions,
        ROUND(MEDIAN(resale_price), 0) AS median_resale_price
    FROM resale_transactions
    GROUP BY town
    ORDER BY transactions DESC
""").fetchdf()

,town,transactions,median_resale_price
0,SENGKANG,19368,525000.0
1,PUNGGOL,17331,535000.0
2,WOODLANDS,17049,470000.0
3,TAMPINES,16579,550000.0
4,YISHUN,16248,435000.0
5,JURONG WEST,15686,462000.0
6,BEDOK,12459,435000.0
7,HOUGANG,12086,500000.0
8,CHOA CHU KANG,10744,495000.0
9,BUKIT BATOK,10232,485000.0


In [11]:
con.close()

In [12]:
import duckdb

con = duckdb.connect(str(DATABASE_PATH))

In [13]:
con.execute("""
    SELECT
        COUNT(*) AS total_transactions
    FROM resale_transactions
""").fetchdf()

,total_transactions
0,239330


In [14]:
con.execute("""
    SELECT
        ROUND(SUM(resale_price), 0) AS total_transaction_value
    FROM resale_transactions
""").fetchdf()

,total_transaction_value
0,1.278730e+11


In [15]:
con.execute("""
    SELECT MEDIAN(resale_price) AS median_resale_price
    FROM resale_transactions
""").fetchdf()

,median_resale_price
0,502000.0


In [16]:
con.execute("""
    SELECT ROUND(MEDIAN(price_per_sqm), 2) AS median_price_per_sqm
    FROM resale_transactions
""").fetchdf()

,median_price_per_sqm
0,5307.69


In [17]:
con.execute("""
    SELECT COUNT(DISTINCT town) AS number_of_towns
    FROM resale_transactions
""").fetchdf()

,number_of_towns
0,26


In [18]:
con.execute("""
    SELECT
        year,
        COUNT(*) AS total_transactions
    FROM resale_transactions
    GROUP BY year
    ORDER BY year
""").fetchdf()

,year,total_transactions
0,2017,20509
1,2018,21561
2,2019,22186
3,2020,23333
4,2021,29087
5,2022,26720
6,2023,25754
7,2024,27832
8,2025,25085
9,2026,17263


In [1]:
import duckdb

con = duckdb.connect(str(DATABASE_PATH))

In [2]:
town_performance = con.execute("""
    SELECT
        town,
        COUNT(*) AS transactions,
        ROUND(MEDIAN(resale_price), 0) AS median_resale_price,
        ROUND(MEDIAN(price_per_sqm), 2) AS median_price_per_sqm,
        ROUND(SUM(resale_price), 0) AS total_transaction_value
    FROM resale_transactions
    GROUP BY town
    ORDER BY transactions DESC
""").fetchdf()

town_performance

,town,transactions,median_resale_price,median_price_per_sqm,total_transaction_value
0,SENGKANG,19368,525000.0,5307.36,1.035542e+10
1,PUNGGOL,17331,535000.0,5731.18,9.544567e+09
2,WOODLANDS,17049,470000.0,4626.87,8.315634e+09
3,TAMPINES,16579,550000.0,5371.90,9.513604e+09
4,YISHUN,16248,435000.0,4908.93,7.421206e+09
5,JURONG WEST,15686,462000.0,4590.16,7.341459e+09
6,BEDOK,12459,435000.0,5159.72,6.097073e+09
7,HOUGANG,12086,500000.0,5241.38,6.388967e+09
8,CHOA CHU KANG,10744,495000.0,4709.09,5.328256e+09
9,BUKIT BATOK,10232,485000.0,5336.75,5.231448e+09


## Town Performance Rankings

### Q1. Which towns have the most transactions?

In [3]:
town_performance[
    ["town", "transactions"]
].sort_values(
    "transactions",
    ascending=False
).head(10)

,town,transactions
0,SENGKANG,19368
1,PUNGGOL,17331
2,WOODLANDS,17049
3,TAMPINES,16579
4,YISHUN,16248
5,JURONG WEST,15686
6,BEDOK,12459
7,HOUGANG,12086
8,CHOA CHU KANG,10744
9,BUKIT BATOK,10232


### Q2. Which towns have the highest median resale prices?

In [4]:
town_performance[
    ["town", "median_resale_price"]
].sort_values(
    "median_resale_price",
    ascending=False
).head(10)

,town,median_resale_price
25,BUKIT TIMAH,784000.0
22,BISHAN,700000.0
17,QUEENSTOWN,675000.0
11,BUKIT MERAH,660000.0
16,PASIR RIS,580000.0
23,CENTRAL AREA,575000.0
15,KALLANG/WHAMPOA,571784.0
3,TAMPINES,550000.0
21,SERANGOON,535000.0
1,PUNGGOL,535000.0


### Q3. Which towns have the highest median price per sqm?

In [5]:
town_performance[
    ["town", "median_price_per_sqm"]
].sort_values(
    "median_price_per_sqm",
    ascending=False
).head(10)

,town,median_price_per_sqm
23,CENTRAL AREA,7686.75
17,QUEENSTOWN,7528.74
11,BUKIT MERAH,6978.02
25,BUKIT TIMAH,6786.89
22,BISHAN,6416.00
24,MARINE PARADE,6338.46
15,KALLANG/WHAMPOA,6271.19
13,TOA PAYOH,5790.26
19,CLEMENTI,5785.12
1,PUNGGOL,5731.18


### Q4. Which towns have the largest total transaction value?

In [6]:
town_performance[
    ["town", "total_transaction_value"]
].sort_values(
    "total_transaction_value",
    ascending=False
).head(10)

,town,total_transaction_value
0,SENGKANG,1.035542e+10
1,PUNGGOL,9.544567e+09
3,TAMPINES,9.513604e+09
2,WOODLANDS,8.315634e+09
4,YISHUN,7.421206e+09
5,JURONG WEST,7.341459e+09
7,HOUGANG,6.388967e+09
6,BEDOK,6.097073e+09
11,BUKIT MERAH,5.882570e+09
8,CHOA CHU KANG,5.328256e+09


In [7]:
import duckdb

con = duckdb.connect(str(DATABASE_PATH))

## Segment Analysis — Town × Flat Type

In [8]:
segment_performance = con.execute("""
    SELECT
        town,
        flat_type,
        COUNT(*) AS transactions,
        ROUND(MEDIAN(resale_price), 0) AS median_resale_price,
        ROUND(MEDIAN(price_per_sqm), 2) AS median_price_per_sqm,
        ROUND(SUM(resale_price), 0) AS total_transaction_value
    FROM resale_transactions
    GROUP BY town, flat_type
    ORDER BY transactions DESC
""").fetchdf()

segment_performance

,town,flat_type,transactions,median_resale_price,median_price_per_sqm,total_transaction_value
0,SENGKANG,4 ROOM,9666,503000.0,5434.78,4.995699e+09
1,PUNGGOL,4 ROOM,9170,530000.0,5706.52,4.991252e+09
2,YISHUN,4 ROOM,7913,448000.0,4782.61,3.569730e+09
3,WOODLANDS,4 ROOM,7715,443000.0,4611.11,3.432545e+09
4,TAMPINES,4 ROOM,6892,535000.0,5376.34,3.783768e+09
...,...,...,...,...,...,...
126,SERANGOON,2 ROOM,19,260000.0,5909.09,4.725868e+06
127,BISHAN,MULTI-GENERATION,13,930000.0,5851.41,1.235878e+07
128,MARINE PARADE,2 ROOM,9,265000.0,6309.52,2.406000e+06
129,BUKIT MERAH,EXECUTIVE,3,750000.0,4807.69,2.357000e+06


### Q1. Which Town × Flat Type segments have the most transactions?

In [9]:
segment_performance[
    ["town", "flat_type", "transactions"]
].sort_values(
    "transactions",
    ascending=False
).head(15)

,town,flat_type,transactions
0,SENGKANG,4 ROOM,9666
1,PUNGGOL,4 ROOM,9170
2,YISHUN,4 ROOM,7913
3,WOODLANDS,4 ROOM,7715
4,TAMPINES,4 ROOM,6892
5,SENGKANG,5 ROOM,6592
6,JURONG WEST,4 ROOM,5878
7,HOUGANG,4 ROOM,5487
8,PUNGGOL,5 ROOM,5429
9,CHOA CHU KANG,4 ROOM,5215


### Q2. Which segments have the highest median resale prices?

In [10]:
segment_performance[
    ["town", "flat_type", "median_resale_price"]
].sort_values(
    "median_resale_price",
    ascending=False
).head(15)

,town,flat_type,median_resale_price
122,QUEENSTOWN,EXECUTIVE,1080000.0
95,CENTRAL AREA,5 ROOM,1068000.0
130,CENTRAL AREA,EXECUTIVE,1034000.0
113,BUKIT TIMAH,EXECUTIVE,1020000.0
88,BISHAN,EXECUTIVE,985000.0
127,BISHAN,MULTI-GENERATION,930000.0
112,ANG MO KIO,EXECUTIVE,920000.0
103,TOA PAYOH,EXECUTIVE,909000.0
69,QUEENSTOWN,5 ROOM,907500.0
111,BUKIT TIMAH,5 ROOM,898000.0


### Q3. Which segments have the highest median price per sqm?

In [11]:
segment_performance[
    ["town", "flat_type", "median_price_per_sqm"]
].sort_values(
    "median_price_per_sqm",
    ascending=False
).head(15)

,town,flat_type,median_price_per_sqm
95,CENTRAL AREA,5 ROOM,9904.76
74,CENTRAL AREA,4 ROOM,9293.04
35,QUEENSTOWN,4 ROOM,8950.96
20,BUKIT MERAH,4 ROOM,8178.00
92,BUKIT BATOK,2 ROOM,7978.72
69,QUEENSTOWN,5 ROOM,7812.32
98,HOUGANG,2 ROOM,7797.88
86,SEMBAWANG,2 ROOM,7765.96
32,KALLANG/WHAMPOA,4 ROOM,7566.23
34,TOA PAYOH,4 ROOM,7540.96


### Q4. Which segments represent the largest total transaction value?

In [12]:
segment_performance[
    ["town", "flat_type", "total_transaction_value"]
].sort_values(
    "total_transaction_value",
    ascending=False
).head(15)

,town,flat_type,total_transaction_value
0,SENGKANG,4 ROOM,4.995699e+09
1,PUNGGOL,4 ROOM,4.991252e+09
5,SENGKANG,5 ROOM,3.809517e+09
4,TAMPINES,4 ROOM,3.783768e+09
2,YISHUN,4 ROOM,3.569730e+09
3,WOODLANDS,4 ROOM,3.432545e+09
8,PUNGGOL,5 ROOM,3.345052e+09
14,TAMPINES,5 ROOM,2.958694e+09
7,HOUGANG,4 ROOM,2.769118e+09
10,WOODLANDS,5 ROOM,2.741687e+09


### Example: Tampines Segment Breakdown

In [13]:
segment_performance[
    segment_performance["town"] == "TAMPINES"
].sort_values(
    "transactions",
    ascending=False
)

,town,flat_type,transactions,median_resale_price,median_price_per_sqm,total_transaction_value
4,TAMPINES,4 ROOM,6892,535000.0,5376.34,3.783768e+09
14,TAMPINES,5 ROOM,4460,645000.0,5211.10,2.958694e+09
21,TAMPINES,3 ROOM,3436,395000.0,5540.54,1.400101e+09
54,TAMPINES,EXECUTIVE,1655,780000.0,5273.97,1.312087e+09
115,TAMPINES,2 ROOM,108,319000.0,6861.70,3.496068e+07
125,TAMPINES,MULTI-GENERATION,28,849000.0,5238.00,2.399189e+07


## Growth Analysis

In [14]:
growth_analysis = con.execute("""
    WITH annual_town_stats AS (
        SELECT
            town,
            year,
            COUNT(*) AS transactions,
            MEDIAN(price_per_sqm) AS median_price_per_sqm
        FROM resale_transactions
        GROUP BY town, year
    ),

    with_previous_year AS (
        SELECT
            town,
            year,
            transactions,
            median_price_per_sqm,

            LAG(transactions) OVER (
                PARTITION BY town
                ORDER BY year
            ) AS previous_year_transactions,

            LAG(median_price_per_sqm) OVER (
                PARTITION BY town
                ORDER BY year
            ) AS previous_year_price_per_sqm

        FROM annual_town_stats
    )

    SELECT
        town,
        year,
        transactions,
        ROUND(median_price_per_sqm, 2) AS median_price_per_sqm,

        ROUND(
            100.0 * (transactions - previous_year_transactions)
            / previous_year_transactions,
            2
        ) AS transaction_growth_pct,

        ROUND(
            100.0 * (median_price_per_sqm - previous_year_price_per_sqm)
            / previous_year_price_per_sqm,
            2
        ) AS price_growth_pct

    FROM with_previous_year

    ORDER BY town, year
""").fetchdf()

growth_analysis

,town,year,transactions,median_price_per_sqm,transaction_growth_pct,price_growth_pct
0,ANG MO KIO,2017,942,4632.35,NaN,NaN
1,ANG MO KIO,2018,1014,4368.70,7.64,-5.69
2,ANG MO KIO,2019,954,4117.65,-5.92,-5.75
3,ANG MO KIO,2020,997,4239.13,4.51,2.95
4,ANG MO KIO,2021,1055,4756.10,5.82,12.20
...,...,...,...,...,...,...
255,YISHUN,2022,1990,5141.52,13.71,10.59
256,YISHUN,2023,1822,5482.90,-8.44,6.64
257,YISHUN,2024,1845,5867.77,1.26,7.02
258,YISHUN,2025,1703,6220.47,-7.70,6.01


### Q1. Which towns had the strongest price growth in 2025?

In [15]:
growth_analysis[
    growth_analysis["year"] == 2025
][
    ["town", "price_growth_pct"]
].sort_values(
    "price_growth_pct",
    ascending=False
).head(10)

,town,price_growth_pct
188,QUEENSTOWN,25.10
238,TOA PAYOH,18.32
78,CENTRAL AREA,11.80
198,SEMBAWANG,10.50
68,BUKIT TIMAH,10.05
58,BUKIT PANJANG,8.58
98,CLEMENTI,8.26
218,SERANGOON,7.96
228,TAMPINES,7.95
158,MARINE PARADE,7.58


### Q2. Which towns had the strongest transaction growth in 2025?

In [16]:
growth_analysis[
    growth_analysis["year"] == 2025
][
    ["town", "transaction_growth_pct"]
].sort_values(
    "transaction_growth_pct",
    ascending=False
).head(10)

,town,transaction_growth_pct
238,TOA PAYOH,31.56
198,SEMBAWANG,17.92
158,MARINE PARADE,14.08
98,CLEMENTI,-0.36
28,BISHAN,-1.57
228,TAMPINES,-2.04
78,CENTRAL AREA,-3.91
108,GEYLANG,-3.91
168,PASIR RIS,-7.17
258,YISHUN,-7.70


### Q3. Which towns experienced both price and transaction growth?

In [17]:
growth_2025 = growth_analysis[
    growth_analysis["year"] == 2025
]

growth_2025[
    (growth_2025["price_growth_pct"] > 0) &
    (growth_2025["transaction_growth_pct"] > 0)
][
    [
        "town",
        "price_growth_pct",
        "transaction_growth_pct"
    ]
].sort_values(
    "price_growth_pct",
    ascending=False
)

,town,price_growth_pct,transaction_growth_pct
238,TOA PAYOH,18.32,31.56
198,SEMBAWANG,10.50,17.92
158,MARINE PARADE,7.58,14.08
